<a href="https://colab.research.google.com/github/asierra383/ScamBusters_Agent/blob/main/Scam_Busters_AI_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

START

In [ ]:
pip install google-adk

In [ ]:
#Import ADK components
import gspread
from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search
from google.genai import types
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters
from google.adk.memory import InMemoryMemoryService
from google.adk.tools import preload_memory
import google.genai as genai

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


In [ ]:
import os
from google.colab import userdata
api_key = userdata.get('GOOGLE_API_KEY')
os.environ['GOOGLE_API_KEY'] = api_key

In [ ]:
#Configure Retry options
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1, # Initial delay before first retry (in seconds)
    http_status_codes=[429, 500, 503, 504] # Retry on these HTTP errors
)

In [ ]:
# 1. Finds the main HYIP monitor sites
discovery_agent = LlmAgent(
    name="DiscoveryAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        api_key=api_key,
        retry_options=retry_config
    ),
    instruction="Find current HYIP monitoring websites. List only the URLs.",
    tools=[google_search],
    output_key="hyip_monitor_list"
)

# 2. Next agent to browse the sites advertised in HYIP monitoring websites and extract content
browser_agent = LlmAgent(
    name="BrowserAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        api_key=api_key,
        retry_options=retry_config),
    instruction="""Given the list of HYIP monitoring websites in {hyip_monitor_list},
    use Google Search to visit each website and extract the cryptocurrency websites it has linked.
    Combine all extracted websites into a single comprehensive string and make
    this combined content available as 'site_content' for the next agent.""",
    tools=[google_search],
    output_key="site_content"
)

print("✅ Discovery Agent and Browser Agent defined.")

✅ Discovery Agent and Browser Agent defined.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# --- STEP 1: SPREADSHEET SETUP ---
import gspread
gc = gspread.service_account(filename='/content/drive/My Drive/Colab Notebooks/scambusters.json')
sh = gc.open("ScamBusters_Tracker")
worksheet = sh.get_worksheet(0)

# --- STEP 2: DEFINE THE TOOL AS A FUNCTION ---
def add_to_spreadsheet(url: str, category: str, notes: str):
    """
    Appends a new website's details to the tracking spreadsheet.
    Use this whenever a new relevant website is discovered.

    Args:
        url (str): The URL of the website to add.
        category (str): The category for the website (e.g., 'HYIP Monitor').
        notes (str): Any additional notes about the website.
    """
    try:
        # Check for duplicates before adding
        existing_urls = worksheet.col_values(1)
        if url in existing_urls:
            return f"Skipped: {url} is already in the sheet."

        worksheet.append_row([url, category, notes])
        return f"Successfully added {url} to the spreadsheet."
    except Exception as e:
        return f"Error updating spreadsheet: {str(e)}"

print("✅ Spreadsheet function defined.")

✅ Spreadsheet function defined.


In [ ]:
# 3. Processes site content and logs new URLs to the spreadsheet
spreadsheet_agent = LlmAgent(
    name="SpreadsheetAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        api_key=api_key,
        retry_options=retry_config
    ),
    instruction="""Analyze the content in {site_content}.
    Identify new, unique HYIP related URLs mentioned in the content.
    For each new unique URL found, use the 'add_to_spreadsheet' tool with category 'HYIP Monitor' and notes 'Discovered by agent'.
    Pass the original {site_content} along to the next agent unchanged.""",
    tools=[add_to_spreadsheet],
    output_key="site_content"
)

print("✅ Spreadsheet Agent defined.")

✅ Spreadsheet Agent defined.


In [ ]:
# 4. Analyzes the extracted text for your specific scam keywords
analyst_agent = LlmAgent(
    name="AnalystAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
    retry_options=retry_config),
    instruction="""Analyze the content in {site_content}.
    Search for scam markers like 'guaranteed high returns' or anonymous teams.
    Provide a final risk assessment.""",
    output_key="final_scam_report"
)

print("✅ Analyst Agent defined.")

✅ Analyst Agent defined.


In [ ]:
# Re-initialize the pipeline with updated agents
scam_pipeline = SequentialAgent(
    name="ScamDetectionPipeline",
    sub_agents=[discovery_agent, browser_agent, spreadsheet_agent, analyst_agent]
)

print("✅ Pipeline updated with agents.")

✅ Pipeline updated with new agent configurations.


In [ ]:
#Run the multi-agent system
runner = InMemoryRunner(agent=scam_pipeline)

print("✅ Runner updated.")

✅ Runner updated.


In [ ]:
#Run by prompting for answer
response = await runner.run_debug(
    "What hyip monitoring websites have recently updated their lists within the past day?"
)


 ### Created new session: debug_session_id

User > What hyip monitoring websites have recently updated their lists within the past day?


_ResourceExhaustedError: 
On how to mitigate this issue, please refer to:

https://google.github.io/adk-docs/agents/models/google-gemini/#error-code-429-resource_exhausted


429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 19.152083772s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generat